# 1. Importation de packages

In [ ]:
import os
import pandas as pd
from utils import charger_donnees_api, imputer_na_par_valeur, tracer_series_temporelles, sauvegarder_donnees_clean, classifier_risque, construire_features, creer_cible_binaire

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

import joblib


# 2. Importation des données

On définit le dossier de données qui n'est créé que s'il n'existe pas encore dans le repertoire de travail.

In [ ]:
DOSSIER_DONNEES = "data/raw"
os.makedirs(DOSSIER_DONNEES, exist_ok=True)

Nous importons ici les données sur les pollens via  l'api open-meteo (**air-quality** pour les données sur les pollens et **archives** pour les données météo); nous avons considéré les données sur un an du 1er janvier 2023 au 01 janvier 2024 pour Rennes.

Données Pollen

In [ ]:
url_pollen = "https://air-quality-api.open-meteo.com/v1/air-quality"

params_pollen = {
    "latitude": 48.11,
    "longitude": -1.67,
    "hourly": ["birch_pollen", "grass_pollen"],
    "start_date": "2021-01-01",
    "end_date": "2025-12-31"
}

fichier_pollen = os.path.join(DOSSIER_DONNEES, "pollen.csv")


df_pol = charger_donnees_api(
    url=url_pollen,
    params=params_pollen,
    fichier_cache=fichier_pollen,
    force_reload= True
)


Données météo

In [ ]:
url_meteo = "https://archive-api.open-meteo.com/v1/archive"

params_meteo = {
    "latitude": 48.11,
    "longitude": -1.67,
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "timezone": "Europe/Paris"
}

fichier_meteo = os.path.join(DOSSIER_DONNEES, "meteo.csv")


df_met = charger_donnees_api(
    url=url_meteo,
    params=params_meteo,
    fichier_cache=fichier_meteo,
    force_reload= True
)

# 3.Fusion et analyse exploratoire

Pour éviter de travailler sur les bases originales, on va travailler sur les copies de ces bases.

In [ ]:
df_meteo = df_met.copy()
df_pollen = df_pol.copy()

## 3.1. Fusion des bases
Pour répondre à la thématique, nous devons fusionner les deux bases; Mais avant  toute fusion, il faut vérifier la structure des deux bases et surtout explorer la clé de fusion qui ici est la date.
Ci-dessous, un aperçu des deux bases:

In [ ]:
pd.concat(
    [df_pollen.head(), df_meteo.head()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

Pour faciliter la lisibilité et la manipulation des variables, il est nécessaire de les renommer.

In [ ]:
df_pollen = df_pollen.rename(columns={
    "birch_pollen": "pollen_bouleau",
    "grass_pollen": "pollen_graminees"
})

df_meteo = df_meteo.rename(columns={
    "temperature_2m": "temperature",
    "precipitation": "precipitations",
    "wind_speed_10m": "vitesse_vent"
})

pd.concat(
    [df_pollen.head(), df_meteo.head()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

Afin de garantir l'intégrité de nos deux bases qui sont temporelles, nous vérifions la régularité du pas de temps via le calcul des différences successives (.diff()). Cette étape confirme la continuité des observations horaires et l'absence de dates manquantes.

In [ ]:
pd.concat(
    [df_pollen["date"].diff().value_counts().head(), df_meteo["date"].diff().value_counts().head()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

Vérifions que les périodes se chevauchent pour les deux bases. ça devrait être le cas au regard des paramètres d'importation mais la vérification en vaut la peine.

In [ ]:
pd.DataFrame({
    "Pollen": [df_pollen["date"].min(), df_pollen["date"].max()],
    "Meteo": [df_meteo["date"].min(), df_meteo["date"].max()]
}, index=["Min date", "Max date"])

On remarque que les données des deux bases sont sans fuseau attachés, ce qui facilite la fusion locale en prenant la date comme clé de fusion.

In [ ]:
print(df_meteo["date"].dt.tz)
print(df_pollen["date"].dt.tz)

On se rend également compte qu'avant la fusion, il n y a pas de valeur manquante dans la base météo.

In [ ]:
pd.concat(
    [df_pollen.isnull().sum(), df_meteo.isnull().sum()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

**Fusion des deux bases**
Fusion des deux bases: avec toutes les vérifications qui viennent d'être faites sur la variable date qui est la clé primaire ici, on  peut fusionner simplement avec un inner_join (correspondances exactes)

In [ ]:
df_pollen_meteo = pd.merge(df_pollen, df_meteo, on="date", how="inner")

df_pollen_meteo.to_csv("data/raw/df_pollen_meteo_merge.csv")

print(df_pollen_meteo.shape)
df_pollen_meteo.head()

## 3.2. Exploration et traitement de la base d'étude

In [ ]:
print("\n--- Informations détaillées (Check-up complet) ---")
df_pollen_meteo.info()

In [ ]:
df_pollen_meteo.isnull().sum()


**Interprétation des valeurs manquantes**

L’analyse des périodes manquantes montre qu’elles correspondent en réalité à des phases biologiques naturelles et non à des erreurs de collecte de données.

Pour le pollen de bouleau, les données sont absentes de janvier à fin février, ce qui correspond à une période de dormance. Elles sont présentes de mars à début août, période de pollinisation, puis redeviennent absentes à partir d’août jusqu'au 20 novembre, marquant la fin de la saison.

Pour les graminées, les données sont également absentes en hiver (janvier-février), présentes de mars à début septembre, ce qui reflète une saison de pollinisation plus longue, puis absentes à nouveau à partir de septembre jusqu'au 20 novembre.

Ces résultats sont cohérents avec les connaissances biologiques et confirment que les valeurs manquantes ne sont pas problématiques, mais traduisent simplement l’absence de pollen hors saison. Cela nous emmene donc à imputer ces valeurs manquantes par des 0.

In [ ]:
from IPython.display import display, HTML
from utils import identifier_plages_manquantes

# Bouleau
display(HTML("<b>Périodes manquantes pour le Pollen de bouleau</b>"))
plages_bouleau = identifier_plages_manquantes(df_pollen_meteo, 'pollen_bouleau')
display(plages_bouleau)

# Graminées
display(HTML("<b>Périodes manquantes pour les graminées</b>"))
plages_graminees = identifier_plages_manquantes(df_pollen_meteo, 'pollen_graminees')
display(plages_graminees)

Bien qu'après le 20 novembre 2023, les données ne soient pas manquantes pour ces deux variables de pollen, les valeurs qu'elles prennent sont nulles comme on peut le voir sur les graphiques plus bas;
Ce même comportement est observé pour les mêmes périodes des années 2024 et 2025;  ce qui est normal vu que la période novembre-decembre n'est pas leur saison. 

In [ ]:
# Définition de la borne temporelle (juste après le dernier NaN)
date_reprise = "2023-11-20 23:00:00"

# Création du sous-ensemble de données pour la fin d'année
df_fin_annee = df_pollen_meteo[df_pollen_meteo['date'] > date_reprise].copy()

# Calcul des statistiques pour se rassurer
stats_fin_annee = df_fin_annee[['pollen_bouleau', 'pollen_graminees']].agg(['max', 'min','mean', 'sum', 'count'])

print(f"--- Analyse de la période : du {df_fin_annee['date'].min()} au {df_fin_annee['date'].max()} ---")
print(f"Nombre total d'heures analysées : {len(df_fin_annee)}")
print("\nStatistiques des pollens sur cette période :")
display(stats_fin_annee)

**Imputation des valeurs manquantes par 0**

In [ ]:
colonnes_pollen = ['pollen_bouleau', 'pollen_graminees']

df_pollen_meteo_clean = imputer_na_par_valeur(df_pollen_meteo, colonnes_pollen)


# 3. Vérification finale
print("Nombre de valeurs manquantes après imputation :")
print(df_pollen_meteo_clean[colonnes_pollen].isna().sum())

In [ ]:
sauvegarder_donnees_clean(
    df=df_pollen_meteo_clean,
    nom_fichier="df_pollen_meteo_clean.csv",
    dossier="data/clean"
)

In [ ]:
tracer_series_temporelles(
    df=df_pollen_meteo_clean,
    variables=[
        "pollen_bouleau",
        "pollen_graminees",
        "temperature",
        "precipitations",
        "vitesse_vent"
    ],
    titres={
        "pollen_bouleau": "Évolution du pollen de bouleau",
        "pollen_graminees": "Évolution du pollen de graminées",
        "temperature": "Évolution de la température",
        "precipitations": "Évolution des précipitations",
        "vitesse_vent": "Évolution de la vitesse du vent"
    },
    ylabels={
        "pollen_bouleau": "Concentration pollen",
        "pollen_graminees": "Concentration pollen",
        "temperature": "Température (°C)",
        "precipitations": "Précipitations (mm)",
        "vitesse_vent": "Vitesse du vent (km/h)"
    },
    couleurs={
        "pollen_bouleau": "green",
        "pollen_graminees": "orange",
        "temperature": "red",
        "precipitations": "blue",
        "vitesse_vent": "purple"
    }
)

## 4.1 Préparation des données

### 4.1.1 Agrégation journalière

Les données étant horaires, elles sont agrégées à l’échelle journalière en retenant le pic maximal de concentration pollinique comme indicateur d’exposition, ainsi que des variables météorologiques résumées par jour, telles que la température moyenne, le cumul des précipitations et la vitesse moyenne du vent.

In [ ]:
# Agrégation journalière
df_jour = df_pollen_meteo_clean.resample('D', on='date').agg(
    pollen_bouleau=('pollen_bouleau', 'max'),
    pollen_graminees=('pollen_graminees', 'max'),
    temperature=('temperature', 'mean'),
    precipitations=('precipitations', 'sum'),
    vitesse_vent=('vitesse_vent', 'mean')
).reset_index()

print(df_jour.shape)
df_jour.head()

### 4.1.2 Création de la variable cible et description

On crée le niveau de risque allergique pour le lendemain, basé sur les seuils du RNSA :
- 0 (Faible) : concentration ≤ 30 grains/m³
- 1 (Modéré) : entre 31 et 80 grains/m³  
- 2 (Élevé) : > 80 grains/m³

In [ ]:
# Variable cible = niveau de risque du lendemain
df_jour['risque_bouleau_j1'] = df_jour['pollen_bouleau'].apply(classifier_risque).shift(-1)
df_jour['risque_graminees_j1'] = df_jour['pollen_graminees'].apply(classifier_risque).shift(-1)

# Supprimer la dernière ligne
df_jour = df_jour.dropna()
df_jour['risque_bouleau_j1'] = df_jour['risque_bouleau_j1'].astype(int)
df_jour['risque_graminees_j1'] = df_jour['risque_graminees_j1'].astype(int)

print(df_jour['risque_bouleau_j1'].value_counts())
print(df_jour['risque_graminees_j1'].value_counts())

In [ ]:
# Distribution sur la saison pollinique du bouleau
df_bouleau_saison = df_jour[df_jour['date'].dt.month.isin([3, 4, 5, 6, 7])]
print("Bouleau - saison pollinique :")
print(df_bouleau_saison['risque_bouleau_j1'].value_counts())
print()

# Distribution sur la saison pollinique des graminées
df_graminees_saison = df_jour[df_jour['date'].dt.month.isin([5, 6, 7, 8, 9])]
print("Graminées - saison pollinique :")
print(df_graminees_saison['risque_graminees_j1'].value_counts())

### 4.1.3 Déséquilibre des classes
La distribution des classes révèle un fort déséquilibre, avec une très forte prédominance de la classe 0 (absence de pic ou de risque faible) pour les deux taxons. Afin de comprendre l’origine de ce déséquilibre et son impact potentiel sur le modèle, nous avons exploré deux contextes temporels distincts :

- un apprentissage sur toute l’année, incluant notamment les périodes hivernales où la concentration pollinique est naturellement très faible ou nulle ;
- un apprentissage restreint à la saison pollinique, c’est‑à‑dire les périodes où l’on attend a priori des épisodes à risque (par exemple, mars à juillet pour le bouleau et mai à septembre pour les graminées).

Les proportions observées sont les suivantes :

|Type| Toute l'année | Saison pollinique |
|--|--|--|
| Bouleau classe 0 | 94% | 87% |
| Graminées classe 0 | 85% | 66% |


Le filtrage sur la saison pollinique améliore légèrement l’équilibre pour les graminées, mais reste insuffisant pour le bouleau, où la classe 0 demeure très majoritaire. Étant donné que notre objectif est de développer un modèle exploitable sur l’ensemble de l’année, nous avons donc choisi de conserver toutes les données, afin de préserver la dynamique réelle de la présence pollinique au cours du cycle saisonnier.

Le déséquilibre est donc traité au niveau de l’apprentissage en utilisant `class_weight='balanced'`, ce qui permet de diminuer l’influence de la classe majoritaire. Par ailleurs, nous privilégions des métriques adaptées comme le **F1‑score macro** plutôt que l’accuracy seule, afin de mieux apprécier la performance sur les classes minoritaires correspondant aux pics de risque.


### 4.1.4 Construction des variables explicatives

Pour prédire le niveau de risque allergique du lendemain, le modèle a besoin d'informations disponibles aujourd'hui. Nous construisons trois types de variables :

**Variables temporelles** : le mois et le jour de l'année permettent au modèle de capturer la saisonnalité — le pollen de bouleau est naturellement plus présent en avril qu'en décembre.

**Variables retardées (lags)** : le niveau de pollen des 3 derniers jours. Si le pollen était élevé hier, il y a de bonnes chances qu'il le soit encore demain. Ces variables capturent l'inertie biologique du phénomène.

**Moyenne glissante sur 3 jours** : lisse les variations brutales et capture la tendance récente de concentration pollinique.

Ces choix s'appuient sur la littérature récente sur la prévision pollinique, qui montre que les variables retardées et les indicateurs de saisonnalité sont parmi les features les plus importantes pour ce type de modèles.

In [ ]:
df_features = construire_features(
    df_jour,
    colonnes_pollen=['pollen_bouleau', 'pollen_graminees']
)
print(df_features.shape)
print(df_features.columns.tolist())
df_features.head()

### 4.1.5 Split temporel train/test

Contrairement à un split aléatoire classique, nous respectons l’ordre chronologique des données, conformément aux bonnes pratiques pour les séries temporelles. Le modèle est entraîné sur la période 2021–2023 et évalué sur la période 2024–2025, de manière à simuler une utilisation prospective dans un contexte opérationnel de prévision pollinique.

In [ ]:
# Définition des features
features = [
    'temperature', 'precipitations', 'vitesse_vent',
    'mois', 'jour_annee',
    'pollen_bouleau_lag1', 'pollen_bouleau_lag2', 'pollen_bouleau_lag3',
    'pollen_bouleau_moy3j',
    'pollen_graminees_lag1', 'pollen_graminees_lag2', 'pollen_graminees_lag3',
    'pollen_graminees_moy3j'
]

# Split temporel
train = df_features[df_features['date'].dt.year <= 2023]
test = df_features[df_features['date'].dt.year >= 2024]

X_train = train[features]
X_test = test[features]

y_train_bouleau = train['risque_bouleau_j1']
y_test_bouleau = test['risque_bouleau_j1']

y_train_graminees = train['risque_graminees_j1']
y_test_graminees = test['risque_graminees_j1']

print("Train :", X_train.shape)
print("Test :", X_test.shape)
print("\nDistribution train - bouleau :")
print(y_train_bouleau.value_counts())
print("\nDistribution train - graminées :")
print(y_train_graminees.value_counts())

## 4.2 Modélisation

### 4.2.1 Pipeline preprocessing + modèles

Nous construisons un pipeline scikit‑learn qui intègre à la fois les étapes de preprocessing et les modèles, afin de garantir une application cohérente aux données d’entraînement et de test. Ce pipeline comprend :

- **StandardScaler** : standardisation des variables continues, nécessaire pour la régression logistique et souvent bénéfique pour d’autres modèles à base de distance.  
- **Trois modèles** : une régression logistique (comme modèle de baseline), un arbre de décision simple et un Random Forest, permettant de comparer la performance entre méthodes simples et ensemblistes.  
- **GridSearchCV** avec `TimeSeriesSplit` : permet de sélectionner les meilleurs hyperparamètres tout en respectant l’ordre chronologique des données, ce qui est cohérent avec notre approche de prévision temporelle.  
- **F1‑macro** comme métrique d’évaluation : adapté au déséquilibre des classes, il permet de mieux apprécier la performance sur les classes minoritaires qu’un simple score de précision.

#### Pipeline avec preprocessing et modèle


In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('modele', LogisticRegression())  # placeholder
])



#### Grille de paramètres pour les 3 modèles


In [ ]:
param_grid = [
    {
        'modele': [LogisticRegression(max_iter=1000)],
        'modele__C': [0.1, 1, 10],
        'modele__class_weight': ['balanced']
    },
    {
        'modele': [DecisionTreeClassifier()],
        'modele__max_depth': [5, 10, None],
        'modele__class_weight': ['balanced']
    },
    {
        'modele': [RandomForestClassifier()],
        'modele__n_estimators': [100, 200],
        'modele__max_depth': [5, 10, None],
        'modele__class_weight': ['balanced']
    }
]



#### Validation croisée temporelle


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)



### 4.2.2 Sélection du meilleur modèle - Bouleau

In [ ]:
# GridSearchCV pour le bouleau
grid_search_bouleau = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_bouleau.fit(X_train, y_train_bouleau)



#### Détails des modèles

In [ ]:
# Tous les modèles avec leurs meilleurs hyperparamètres - Bouleau
resultats_bouleau = pd.DataFrame(grid_search_bouleau.cv_results_)
resultats_bouleau['modele'] = resultats_bouleau['param_modele'].apply(
    lambda x: type(x).__name__
)

# Meilleur résultat par modèle avec ses paramètres
idx_meilleurs = resultats_bouleau.groupby('modele')['mean_test_score'].idxmax()
resume_bouleau = resultats_bouleau.loc[idx_meilleurs, [
    'modele', 'mean_test_score', 'params'
]].sort_values('mean_test_score', ascending=False).round(3)

print("=== Bouleau - Comparaison des modèles ===")
# Afficher les paramètres complets
for _, row in resume_bouleau.iterrows():
    print(f"\n{row['modele']} (F1-macro: {row['mean_test_score']:.3f})")
    print(row['params'])

### 4.2.3 Sélection du meilleur modèle - Graminées

In [ ]:
# GridSearchCV pour les graminées
grid_search_graminees = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_graminees.fit(X_train, y_train_graminees)



#### Détails des modèles

In [ ]:
# Tous les modèles avec leurs meilleurs hyperparamètres - Graminees
resultats_graminees = pd.DataFrame(grid_search_graminees.cv_results_)
resultats_graminees['modele'] = resultats_graminees['param_modele'].apply(
    lambda x: type(x).__name__
)

# Meilleur résultat par modèle avec ses paramètres
idx_meilleurs = resultats_graminees.groupby('modele')['mean_test_score'].idxmax()
resume_graminees = resultats_graminees.loc[idx_meilleurs, [
    'modele', 'mean_test_score', 'params'
]].sort_values('mean_test_score', ascending=False).round(3)

print("=== Graminees - Comparaison des modèles ===")
# Afficher les paramètres complets
for _, row in resume_graminees.iterrows():
    print(f"\n{row['modele']} (F1-macro: {row['mean_test_score']:.3f})")
    print(row['params'])

Le GridSearchCV teste 12 combinaisons d'hyperparamètres pour 3 modèles,évaluées par validation croisée temporelle (5 folds, F1-macro).

**Bouleau**

| Modèle | F1-macro (CV) | Hyperparamètres optimaux |
|---|---|---|
| Random Forest | **0.853** | max_depth=5, n_estimators=100 |
| Arbre de décision | 0.805 | max_depth=None |
| Régression logistique | 0.601 | C=10 |

**Graminées**

| Modèle | F1-macro (CV) | Hyperparamètres optimaux |
|---|---|---|
| Random Forest | **0.834** | max_depth=10, n_estimators=200 |
| Arbre de décision | 0.719 | max_depth=5 |
| Régression logistique | 0.646 | C=10 |

Le Random Forest est le meilleur modèle pour les deux pollens. La régression logistique performe nettement moins bien, ce qui suggère que la relation entre météo et risque pollinique est non-linéaire.
Les meilleurs modèles (Random Forest pour le bouleau et les graminées) sont retenus pour l’évaluation finale sur le jeu de test.

## 4.3 Évaluation sur le jeu de test

On évalue les meilleurs modèles sélectionnés sur le jeu de test(2024-2025), données jamais vues pendant l'entraînement.

### 4.3.1 Rapport de classification 


In [ ]:
# Prédictions sur le jeu de test
y_pred_bouleau = grid_search_bouleau.predict(X_test)
y_pred_graminees = grid_search_graminees.predict(X_test)

# Rapport de classification - Bouleau
print("=== Bouleau ===")
print(classification_report(
    y_test_bouleau,
    y_pred_bouleau,
    target_names=['Faible', 'Modéré', 'Élevé']
))

# Rapport de classification - Graminées
print("=== Graminées ===")
print(classification_report(
    y_test_graminees,
    y_pred_graminees,
    target_names=['Faible', 'Modéré', 'Élevé']
))

Le modèle retenu (Random Forest) obtient une bonne performance globale, avec une accuracy élevée de 0.93 pour le bouleau et 0.95 pour les graminées. Cependant, compte tenu du déséquilibre des classes, l’interprétation fine par classe est plus informative.

Pour le bouleau, le modèle détecte très bien les jours à faible risque, mais reste imparfait sur les classes modérée et élevée, notamment à cause d’un rappel acceptable mais d’une faible précision pour la classe modérée. Pour les graminées, le modèle est globalement plus performant, notamment pour les classes modérée et élevée, tout en maintenant une excellente détection des jours à faible risque.

Ces résultats montrent que le modèle est capable d’identifier les jours à risque allergique (modéré et élevé), mais avec une tendance à sur‑prédire ces niveaux, ce qui est cohérent avec le fort déséquilibre des classes et la difficulté à caractériser précisément les pics de courte durée.

### 4.3.2 Matrices de confusion

Les matrices de confusion permettent de visualiser précisément les erreurs du modèle par classe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = ['Faible', 'Modéré', 'Élevé']

for ax, y_test, y_pred, titre in zip(
    axes,
    [y_test_bouleau, y_test_graminees],
    [y_pred_bouleau, y_pred_graminees],
    ['Bouleau', 'Graminées']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=labels, yticklabels=labels, ax=ax
    )
    ax.set_title(f'Matrice de confusion - {titre}')
    ax.set_ylabel('Valeur réelle')
    ax.set_xlabel('Valeur prédite')

plt.tight_layout()
plt.show()

**Bouleau :**
- Faible : très bien prédit (662/697 corrects), avec quelques confusions avec la classe Modéré (29 cas).
- Modéré : partiellement prédit (7/15 corrects), souvent confondu avec la classe Élevé (8 cas).
- Élevé : correctement prédit (12/18 corrects), avec quelques confusions avec la classe Modéré (6 cas).

**Graminées :**
- Faible : excellente prédiction (617/630 corrects).
- Modéré : bien prédit (63/84 corrects), avec quelques confusions avec la classe Élevé (14 cas).
- Élevé : partiellement prédit (11/16 corrects), principalement confondu avec la classe Modéré (5 cas).

**Interprétation globale :**
Les confusions entre classes voisines (Modéré/Élevé) sont largement dominantes, ce qui est cohérent avec la nature graduelle de la frontière entre ces niveaux de risque, les seuils étant par définition arbitraires sur une échelle continue de concentration pollinique. C'est l'une des motivations pour tester une classification binaire (Faible / À risque).

### 4.3.3 Importance des variables

On examine quelles variables ont le plus contribué aux décisions du Random Forest. Cela permet de valider nos choix de features et d'interpréter le modèle.

##### Extraction du meilleur modèle Random Forest


In [ ]:
best_model_bouleau = grid_search_bouleau.best_estimator_.named_steps['modele']
best_model_graminees = grid_search_graminees.best_estimator_.named_steps['modele']

#### Importance des variables

In [ ]:
importances_bouleau = pd.Series(
    best_model_bouleau.feature_importances_,
    index=features
).sort_values(ascending=True)

importances_graminees = pd.Series(
    best_model_graminees.feature_importances_,
    index=features
).sort_values(ascending=True)


#### Visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

importances_bouleau.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Importance des variables - Bouleau')
axes[0].set_xlabel('Importance')

importances_graminees.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Importance des variables - Graminées')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

**Limite :**

Ces résultats nuancent notre problématique initiale. Nous supposions que les conditions météorologiques seraient les principaux déterminants du risque allergique. Or le modèle montre que l’historique récent du pollen (lags et moyenne glissante) est bien plus prédictif que la météo seule.

Cela soulève une question méthodologique et opérationnelle : dans un contexte de prévision réelle, l’historique pollinique n’est pas toujours disponible à l’avance. Un modèle basé uniquement sur la météo serait plus utile opérationnellement, mais probablement moins performant.

Cette limite ouvre deux perspectives :
1. Tester un modèle avec uniquement les variables météorologiques pour mesurer la perte de performance par rapport au modèle actuel.
2. Conserver le modèle actuel en acceptant qu’il nécessite des données polliniques récentes comme input, tout en discutant les implications pratiques (disponibilité des mesures, délais, etc.).

### 4.3.4 Limites et perspectives

Plusieurs limites doivent être soulignées :

**Limites liées aux données :**
- Les classes minoritaires (Modéré et Élevé) sont sous-représentées, notamment pour le bouleau (15 cas Modéré dans le test), ce qui rend l'apprentissage difficile
- Les données couvrent une seule ville (Rennes), limitant la généralisation à d'autres territoires

**Limites liées à la modélisation :**
- L'importance des variables révèle que l'historique pollinique prédit mieux le risque que la météo seule, ce qui nuance notre problématique initiale centrée sur les conditions météorologiques
- Une chute du F1-macro entre la validation croisée et le test pour le bouleau (0.853 → 0.59) suggère un possible data drift entre les années d'entraînement et de test

**Perspectives :**
Pour améliorer ces résultats, nous explorons deux pistes dans les sections suivantes :
1. Une classification binaire (Faible / À risque) pour améliorer la détection des épisodes polliniques
2. Un modèle basé uniquement sur la météo pour mesurer l'apport réel de l'historique pollinique

## 4.4 Amélioration : classification binaire

Face aux difficultés du modèle sur les classes minoritaires, nous testons une reformulation binaire : Faible (0) vs À risque (1), en regroupant Modéré et Élevé en une seule classe.

In [ ]:
df_binaire = creer_cible_binaire(
    df_features,
    colonnes_pollen=['pollen_bouleau', 'pollen_graminees']
)

print(df_binaire['risque_bin_pollen_bouleau_j1'].value_counts())
print(df_binaire['risque_bin_pollen_graminees_j1'].value_counts())

Les classes sont toujours déséquilibrées mais c'est mieux qu'avant :

| Type| Avant (3 classes) | Après (binaire) |
|---|---|---|
| Bouleau classe majoritaire | 94% | 94% |
| Bouleau classes minoritaires | 2.5% + 3.5% | **6%** |
| Graminées classe majoritaire | 85% | 86% |
| Graminées classes minoritaires | 10% + 4% | **14%** |


### Split et GridSearchCV

In [ ]:
# Split temporel
train_bin = df_binaire[df_binaire['date'].dt.year <= 2023]
test_bin = df_binaire[df_binaire['date'].dt.year >= 2024]

X_train_bin = train_bin[features]
X_test_bin = test_bin[features]

y_train_bouleau_bin = train_bin['risque_bin_pollen_bouleau_j1']
y_test_bouleau_bin = test_bin['risque_bin_pollen_bouleau_j1']

y_train_graminees_bin = train_bin['risque_bin_pollen_graminees_j1']
y_test_graminees_bin = test_bin['risque_bin_pollen_graminees_j1']

# GridSearchCV - Bouleau binaire
grid_search_bouleau_bin = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_search_bouleau_bin.fit(X_train_bin, y_train_bouleau_bin)
print("Meilleur modèle bouleau :", grid_search_bouleau_bin.best_params_['modele'])
print("Meilleur F1-macro (CV) :", round(grid_search_bouleau_bin.best_score_, 3))

# GridSearchCV - Graminées binaire
grid_search_graminees_bin = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_search_graminees_bin.fit(X_train_bin, y_train_graminees_bin)
print("Meilleur modèle graminées :", grid_search_graminees_bin.best_params_['modele'])
print("Meilleur F1-macro (CV) :", round(grid_search_graminees_bin.best_score_, 3))

### Evaluation sur les données de test

In [ ]:
# Prédictions
y_pred_bouleau_bin = grid_search_bouleau_bin.predict(X_test_bin)
y_pred_graminees_bin = grid_search_graminees_bin.predict(X_test_bin)

# Rapports
print("=== Bouleau binaire ===")
print(classification_report(
    y_test_bouleau_bin,
    y_pred_bouleau_bin,
    target_names=['Faible', 'À risque']
))

print("=== Graminées binaire ===")
print(classification_report(
    y_test_graminees_bin,
    y_pred_graminees_bin,
    target_names=['Faible', 'À risque']
))

In [ ]:
# F1-macro sur train vs test
f1_train_bouleau = f1_score(y_train_bouleau_bin, 
                            grid_search_bouleau_bin.best_estimator_.predict(X_train_bin), 
                            average='macro')
f1_test_bouleau = f1_score(y_test_bouleau_bin,
                           y_pred_bouleau_bin,
                           average='macro')

f1_train_graminees = f1_score(y_train_graminees_bin,
                              grid_search_graminees_bin.best_estimator_.predict(X_train_bin),
                              average='macro')
f1_test_graminees = f1_score(y_test_graminees_bin,
                             y_pred_graminees_bin,
                             average='macro')

print("Bouleau - Train F1-macro :", round(f1_train_bouleau, 3))
print("Bouleau - Test F1-macro  :", round(f1_test_bouleau, 3))
print()
print("Graminées - Train F1-macro :", round(f1_train_graminees, 3))
print("Graminées - Test F1-macro  :", round(f1_test_graminees, 3))

###  Analyse du surapprentissage

| Type| Train F1-macro | Test F1-macro | Écart |
|---|---|---|---|
| Bouleau | 1,000 | 0,883 | 0,117 |
| Graminées | 0,956 | 0,945 | 0,011 |

Le bouleau présente un surapprentissage notable : le modèle atteint un F1‑macro parfait sur le jeu d’entraînement (1.000), mais chute à 0,883 sur le jeu de test. Cela s’explique notamment par le très faible nombre de cas “à risque” dans les données d’entraînement (99 cas sur 1092), ce qui rend l’apprentissage instable et favorise la sur‑adaptation aux exemples rares.

Les graminées, en revanche, ne présentent quasiment pas de surapprentissage (écart de 0.011), ce qui confirme que le modèle généralise bien sur ce pollen, mieux représenté dans les données.

Pour réduire le surapprentissage du bouleau, on pourrait contraindre davantage le modèle, par exemple via des hyperparamètres comme `min_samples_leaf` ou `max_depth`, dans de futurs travaux.

## 4.5 Modèle basé uniquement sur la météo

L'importance des variables a montré que l'historique pollinique domine les prédictions. Nous testons ici un modèle utilisant uniquement les variables météo et la saisonnalité, sans historique pollinique, pour mesurer leur apport réel.

In [ ]:
# Features météo uniquement
features_meteo = [
    'temperature', 'precipitations', 'vitesse_vent',
    'mois', 'jour_annee'
]

# Split temporel
X_train_meteo = train[features_meteo]
X_test_meteo = test[features_meteo]

# GridSearchCV - Bouleau météo seul
grid_search_bouleau_meteo = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_search_bouleau_meteo.fit(X_train_meteo, y_train_bouleau)
print("Bouleau météo - Meilleur modèle :", 
      grid_search_bouleau_meteo.best_params_['modele'])
print("Bouleau météo - F1-macro (CV) :", 
      round(grid_search_bouleau_meteo.best_score_, 3))

# GridSearchCV - Graminées météo seul
grid_search_graminees_meteo = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_search_graminees_meteo.fit(X_train_meteo, y_train_graminees)
print("Graminées météo - Meilleur modèle :", 
      grid_search_graminees_meteo.best_params_['modele'])
print("Graminées météo - F1-macro (CV) :", 
      round(grid_search_graminees_meteo.best_score_, 3))

### Evaluation sur les données de test

In [ ]:
# Prédictions météo seul
y_pred_bouleau_meteo = grid_search_bouleau_meteo.predict(X_test_meteo)
y_pred_graminees_meteo = grid_search_graminees_meteo.predict(X_test_meteo)

print("=== Bouleau - Météo seule ===")
print(classification_report(
    y_test_bouleau,
    y_pred_bouleau_meteo,
    target_names=['Faible', 'Modéré', 'Élevé']
))

print("=== Graminées - Météo seule ===")
print(classification_report(
    y_test_graminees,
    y_pred_graminees_meteo,
    target_names=['Faible', 'Modéré', 'Élevé']
))

## Essai avec la variable binaire et les variables explicatives météorologiques

In [ ]:
X_train_meteo_bin = train_bin[features_meteo]
X_test_meteo_bin = test_bin[features_meteo]

print("Train météo bin :", X_train_meteo_bin.shape)
print("Test météo bin :", X_test_meteo_bin.shape)

In [ ]:
# Refaire le GridSearchCV météo binaire avec les bonnes données
grid_search_bouleau_meteo_bin = GridSearchCV(
    pipeline, param_grid, cv=tscv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_bouleau_meteo_bin.fit(X_train_meteo_bin, y_train_bouleau_bin)

grid_search_graminees_meteo_bin = GridSearchCV(
    pipeline, param_grid, cv=tscv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_graminees_meteo_bin.fit(X_train_meteo_bin, y_train_graminees_bin)

# Prédictions avec X_test_meteo_bin
y_pred_bouleau_meteo_bin = grid_search_bouleau_meteo_bin.predict(X_test_meteo_bin)
y_pred_graminees_meteo_bin = grid_search_graminees_meteo_bin.predict(X_test_meteo_bin)

print("=== Bouleau - Météo seule binaire ===")
print(classification_report(
    y_test_bouleau_bin, y_pred_bouleau_meteo_bin,
    target_names=['Faible', 'À risque']
))

print("=== Graminées - Météo seule binaire ===")
print(classification_report(
    y_test_graminees_bin, y_pred_graminees_meteo_bin,
    target_names=['Faible', 'À risque']
))

### 4.5.3 Tableau comparatif final

| Approche | Bouleau F1-macro | Graminées F1-macro |
|---|---|---|
| Météo seule (3 classes) | 0.52 | 0.72 |
| Complet (3 classes) | 0.59 | 0.76 |
| Météo seule binaire | 0.79 | 0.90 |
| Complet binaire | 0.88 | 0.94 |

L'historique pollinique apporte un gain modeste (+0.09 bouleau, +0.04 graminées). La simplification binaire est le facteur d'amélioration le plus important. 

En pratique, un modèle météo seul binaire (0.79/0.90) serait plus utile opérationnellement car il ne nécessite pas de données polliniques récentes comme input.

In [ ]:
# F1-macro train vs test - Météo seule binaire
f1_train_bouleau_meteo_bin = f1_score(
    y_train_bouleau_bin,
    grid_search_bouleau_meteo_bin.predict(X_train_meteo_bin),
    average='macro'
)
f1_train_graminees_meteo_bin = f1_score(
    y_train_graminees_bin,
    grid_search_graminees_meteo_bin.predict(X_train_meteo_bin),
    average='macro'
)

print("Bouleau météo bin - Train F1 :", round(f1_train_bouleau_meteo_bin, 3))
print("Bouleau météo bin - Test F1  :", round(f1_score(y_test_bouleau_bin, y_pred_bouleau_meteo_bin, average='macro'), 3))
print()
print("Graminées météo bin - Train F1 :", round(f1_train_graminees_meteo_bin, 3))
print("Graminées météo bin - Test F1  :", round(f1_score(y_test_graminees_bin, y_pred_graminees_meteo_bin, average='macro'), 3))


| Approche | Train F1 | Test F1 | Écart |
|---|---|---|---|
| Complet 3 classes - Bouleau | 1.000 | 0.590 | 0.410 |
| Complet 3 classes - Graminées | 0.956 | 0.760 | 0.196 |
| Complet binaire - Bouleau | 1.000 | 0.883 | 0.117 |
| Complet binaire - Graminées | 0.956 | 0.945 | 0.011 |
| Météo binaire - Bouleau | 0.859 | 0.787 | 0.072 |
| Météo binaire - Graminées | 0.943 | 0.901 | 0.042 |

**Interprétation :**

Le modèle météo seul binaire présente les écarts train/test les plus faibles pour le bouleau (0.072), indiquant moins de surapprentissage que le modèle complet (0.117). 

Le meilleur compromis performance/surapprentissage est le **modèle météo seul binaire** : performances solides (0.787/0.901) sanssurapprentissage excessif, et utilisable sans données polliniques récentes.

## 4.6 Conclusion générale

Ce projet avait pour objectif d’anticiper les pics de concentration pollinique à Rennes à partir des conditions météorologiques.

**Ce qu’on a appris :**

1. **La classification binaire est plus adaptée** que les 3 classes pour ce type de données fortement déséquilibrées. La rareté des épisodes polliniques intenses rend difficile l’apprentissage des classes minoritaires, ce qui justifie la simplification en deux niveaux de risque (faible / à risque).

2. **L’historique pollinique améliore les prédictions** mais de façon modeste (+0.09 pour le bouleau, +0.04 pour les graminées). La météo seule capture déjà l’essentiel de l’information, ce qui souligne la robustesse des relations météo–risque allergique.

3. **Le Random Forest est systématiquement le meilleur modèle** parmi les trois testés, confirmant que la relation entre météo et risque pollinique est non‑linéaire, et que les méthodes de type forêts d’arbres sont bien adaptées à ce type de tâche.

4. **Le modèle météo seule binaire offre le meilleur compromis** : performances solides, moins de surapprentissage, et utilisable sans données polliniques récentes — ce qui le rend plus opérationnel en pratique, notamment dans un contexte de prévision en temps réel.

**Perspectives :**
- Étendre l’analyse à d’autres villes françaises, afin de tester la robustesse et la transférabilité du modèle.
- Intégrer des variables supplémentaires (humidité, ensoleillement, indice de végétation, etc.) pour affiner la description des conditions météorologiques.
- Tester des modèles séquentiels (par exemple des architectures LSTM) pour mieux exploiter la structure temporelle des données et potentiellement améliorer la prévision à plusieurs jours d’avance.

## 4.7 Sauvegarde des modèles

On sauvegarde les meilleurs modèles pour une utilisation future notamment dans une application de prévision du risque allergique.

In [ ]:

os.makedirs("models", exist_ok=True)

# Sauvegarde des meilleurs modèles (météo seul binaire)
joblib.dump(
    grid_search_bouleau_meteo_bin.best_estimator_,
    "models/modele_bouleau.pkl"
)
joblib.dump(
    grid_search_graminees_meteo_bin.best_estimator_,
    "models/modele_graminees.pkl"
)

# Sauvegarde des features utilisées
joblib.dump(features_meteo, "models/features_meteo.pkl")

print("Modèles sauvegardés :")
print(os.listdir("models"))